# 14 - Tool Calling

This notebook starts the Agentic AI track.

Your supplied agent notebook defines the progression:

```text
LLM → Tools → Tool Schema → Tool Binding
    → Tool Call → Tool Execution
    → ToolMessage → Final Answer → Agent Loop
```

The core idea is simple:

> The model **requests** an action; your application **executes** the action.

This separation is the foundation of safe agent systems.

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# Optional packages:
# %pip install -qU langchain-core langchain-openai pydantic python-dotenv

## 1. What is a tool?

A tool is an external capability exposed to the model.

Examples:
- calculator
- database lookup
- web search
- weather
- file operation
- API call

Without tools:
```text
user → model → text
```

With tools:
```text
user → model → tool call → application executes → result → model
```

In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    'Evaluate a simple arithmetic expression for educational use.'
    allowed = set('0123456789+-*/(). %')
    if not set(expression) <= allowed:
        return 'Unsupported expression'
    try:
        return str(eval(expression, {'__builtins__': {}}, {}))
    except Exception as exc:
        return f'Tool error: {exc}'

print(calculator.name)
print(calculator.description)
print(calculator.args_schema.model_json_schema())

## 2. Tool schema

The model needs:
- name
- description
- input fields
- types
- required/optional fields

The schema is a contract between the model and application.

In [ ]:
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    city: str = Field(description='City name')
    unit: str = Field(default='c', description='Temperature unit: c or f')

print(WeatherInput.model_json_schema())

## 3. Bind tools to a chat model

Binding tells the model which capabilities are available.

This notebook keeps the live call optional because a model/API key is environment-specific.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

if CHAT_MODEL:
    model = ChatOpenAI(model=CHAT_MODEL, temperature=0)
    model_with_tools = model.bind_tools([calculator])

    response = model_with_tools.invoke([
        HumanMessage(content='Calculate 125 * 8 using the calculator.')
    ])

    print('content:', response.content)
    print('tool_calls:', response.tool_calls)
else:
    print('Configure OPENAI_CHAT_MODEL to run the live example.')

## 4. Execute a tool call

A tool call is data. The application reads the name and arguments, selects a trusted implementation, and executes it.

In [ ]:
def execute_tool_call(call, tools):
    tool_map = {t.name: t for t in tools}
    selected = tool_map.get(call['name'])
    if selected is None:
        raise ValueError(f'Unknown tool: {call["name"]}')
    return selected.invoke(call['args'])

example_call = {
    'name': 'calculator',
    'args': {'expression': '125 * 8'},
    'id': 'call-1',
}

print(execute_tool_call(example_call, [calculator]))

## 5. AIMessage and ToolMessage

The common message flow is:

```text
HumanMessage
     ↓
AIMessage(tool_calls)
     ↓
ToolMessage(result)
     ↓
AIMessage(final answer)
```

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage

human = HumanMessage(content='What is 2 + 2?')

ai = AIMessage(
    content='',
    tool_calls=[
        {'name': 'calculator', 'args': {'expression': '2 + 2'}, 'id': 'call-1'}
    ]
)

tool = ToolMessage(content='4', tool_call_id='call-1')

print(type(human).__name__)
print(type(ai).__name__, ai.tool_calls)
print(type(tool).__name__, tool.content)

## 6. Safety note

Never let the model directly access arbitrary Python, shell, SQL, filesystem, or network capabilities.

Instead:
```text
model request
   ↓
validation / authorization
   ↓
approved tool
   ↓
execution
```

In [ ]:
ALLOWED_TOOLS = {'calculator'}

def allowed_tool(name):
    return name in ALLOWED_TOOLS

print(allowed_tool('calculator'))
print(allowed_tool('shell'))

## 7. Practice

Build two tools: `calculator` and `word_count`. Then test which tool should be selected for five different questions.

## Evaluation

Evaluate:
- tool selection accuracy
- argument accuracy
- tool execution success
- unknown-tool rejection
- invalid-argument behavior

Improvement:
- better descriptions
- typed schemas
- validation
- least privilege
- explicit tool allowlists.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/

The detailed tool/schema/AIMessage/ToolMessage flow is directly grounded in your supplied `AI_Agent_Using_Tools_CampusX_Consolidated(1).ipynb`. fileciteturn19file0